In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import pandas as pd

transform = transforms.Compose([
    transforms.ToPILImage(), # Convert NumPy array to PIL Image
    transforms.Resize((32, 32), antialias=True),
    transforms.RandomRotation(10),
    transforms.ToTensor(), # Convert PIL Image to Tensor and normalize to [0, 1]
])

# Load MNIST dataset
test_df = pd.read_csv('/content/sign_mnist_test.csv')
train_df = pd.read_csv('/content/sign_mnist_train.csv')

In [ ]:
import torch
from torch.utils.data import Dataset
import numpy as np

class MNISTDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # 1. Get the RAW INTEGER label from the dataframe
        label = int(row.iloc[0])

        # 2. MATHEMATICAL FIX: Shift the integer to close the gap left by 'J' (9)
        # This makes our labels perfectly continuous from 0 to 23,
        # preventing the "CUDA error: device-side assert triggered" crash!
        if label > 9:
            label -= 1

        # 3. Extract and reshape the image
        # We use 'uint8' because PyTorch's transforms.ToTensor() expects it
        # in order to automatically convert it to a float and divide by 255.0
        image = row[1:].values.astype(np.uint8).reshape(28, 28)

        # 4. Apply your transformations (ToPILImage, Resize, ToTensor, etc.)
        if self.transform:
            image = self.transform(image)
        else:
            # Fallback just in case you don't pass a transform
            image = torch.tensor(image, dtype=torch.float32).unsqueeze(0) / 255.0

        # Return the transformed image and the corrected integer label
        return image, label

In [ ]:
train_dataset = MNISTDataset(train_df, transform=transform)
test_dataset = MNISTDataset(test_df, transform=transform)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class AdvancedSignLanguageModel(nn.Module):
    def __init__(self, num_classes=24):
        super(AdvancedSignLanguageModel, self).__init__()

        self.densenet = models.densenet121(weights=None)
        self.densenet.features.conv0 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=3, # Reduced from 7
            stride=1,      # Reduced from 2
            padding=1,
            bias=False
        )

        # Remove the initial MaxPool layer entirely to preserve 28x28 spatial resolution
        self.densenet.features.pool0 = nn.Identity()

        # 3. Modify the Final Layer (Output)
        # DenseNet outputs 1000 classes by default. We change it to our 24 sign language classes.
        num_ftrs = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.densenet(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = AdvancedSignLanguageModel(num_classes=24).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 1. Setup Device & Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = AdvancedSignLanguageModel(num_classes=24).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# 2. Lists to track metrics for the final graphs
epochs = 15
epoch_losses = []
epoch_accuracies = []

# 3. Training Loop (Train Set Only)
print("Starting Training...")
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct_train = 0
    total_train = 0
    current_lr = scheduler.get_last_lr()[0]

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)

        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct_train / total_train

    epoch_losses.append(avg_loss)
    epoch_accuracies.append(accuracy)

    print(f"Epoch [{epoch+1}/{epochs}] | LR: {current_lr:.6f} | Train Loss: {avg_loss:.4f} | Train Acc: {accuracy:.2f}%")


In [ ]:


# Create a figure with 2 subplots (side by side)
plt.figure(figsize=(12, 5))

# Plot 1: Training Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), epoch_losses, marker='o', color='red', label='Training Loss')
plt.title('Training Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

# Plot 2: Training Accuracy
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), epoch_accuracies, marker='o', color='blue', label='Training Accuracy')
plt.title('Training Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

# Adjust layout and save/show the plot
plt.tight_layout()
plt.savefig('training_metrics.png')  # Saves the image to your files
plt.show()  # Displays the image in Jupyter/Colab

#TEST

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

correct_test = 0
total_test = 0

print(f"Evaluating model on {len(test_dataset)} test images...")

with torch.no_grad():
    for images, labels in test_loader:

        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()
test_accuracy = 100 * correct_test / total_test

print(f"FINAL TEST ACCURACY: {test_accuracy:.2f}%")